# Word Embeddings for Document Classification : Practice & Experiments

**Authored by Alexandre Mathias DONNAT, Sr - Télécom Paris**

In this lab work, we will work to construct our own graph neural network using PyTorch Geometric (PyG) and then apply that model on two Open Graph Benchmark (OGB) datasets. These two datasets will be used to benchmark our model's performance on two different graph-based tasks: 1) node property prediction, predicting properties of single nodes and 2) graph property prediction, predicting properties of entire graphs or subgraphs.

First, we will learn how PyTorch Geometric stores graphs as PyTorch tensors.

Then, we will load and inspect one of the Open Graph Benchmark (OGB) datasets by using the `ogb` package. OGB is a collection of realistic, large-scale, and diverse benchmark datasets for machine learning on graphs. The `ogb` package not only provides data loaders for each dataset but also model evaluators.

Lastly, we will build our own graph neural network using PyTorch Geometric. We will then train and evaluate our model on the OGB node property prediction and graph property prediction tasks.

**Note**: Make sure to **sequentially run all the cells in each section**, so that the intermediate variables / packages will carry over to the next cell.

We recommend we save a copy of this colab in our drive so our don't lose progress!

**IMPORTANT**: This lab session is not graded, but some points here might appear in the final exam.

# Device
We might need to use a GPU for this Colab to run quickly.

Please click `Runtime` and then `Change runtime type`. Then set the `hardware accelerator` to **GPU**.

# Setup
The installation of PyG on Colab can be a little bit tricky. First let us check which version of PyTorch we are running

In [1]:
import torch
import os
print("PyTorch has version {}".format(torch.__version__))

PyTorch has version 2.10.0+cpu


Download the necessary packages for PyG. Make sure that our version of torch matches the output from the cell above. In case of any issues, more information can be found on the [PyG's installation page](https://pytorch-geometric.readthedocs.io/en/latest/notes/installation.html).

In [2]:
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.10.0+cu128.html
!pip install ogb

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html


# Important
CUDA and PyTorch versions in Google Colab are most likely outdated. For a real job, please use an actual IDE (like PyCharm or VS Code) and install the latest versions of PyTorch and PyTorch geometric.

# 1) PyTorch Geometric (Datasets and Data)


PyTorch Geometric has two classes for storing and/or transforming graphs into tensor format. One is `torch_geometric.datasets`, which contains a variety of common graph datasets. Another is `torch_geometric.data`, which provides the data handling of graphs in PyTorch tensors.

In this section, we will learn how to use `torch_geometric.datasets` and `torch_geometric.data` together.

## PyG Datasets

The `torch_geometric.datasets` class has many common graph datasets. Here we will explore its usage through one example dataset.

In [3]:
from torch_geometric.datasets import TUDataset

root = './enzymes'
name = 'ENZYMES'

# The ENZYMES dataset
pyg_dataset= TUDataset(root, name)

# there are 600 graphs in this dataset
print(pyg_dataset)

c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\libpyg.pyd
  import torch_geometric.typing
c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_scatter\_scatter_cuda.pyd
  import torch_geometric.typing
c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_cluster\_grid_cuda.pyd
  import torch_geomet

ENZYMES(600)


## Question 1: What is the number of classes and number of features in the ENZYMES dataset?

In [4]:
def get_num_classes(pyg_dataset):
  # returns number of classes
  num_classes = pyg_dataset.num_classes
  return num_classes

def get_num_features(pyg_dataset):
  # returns number of node features
  num_features = pyg_dataset.num_features
  return num_features

num_classes = get_num_classes(pyg_dataset)
num_features = get_num_features(pyg_dataset)

print("{} dataset has {} classes".format(name, num_classes))
print("{} dataset has {} features".format(name, num_features))

ENZYMES dataset has 6 classes
ENZYMES dataset has 3 features


## PyG Data

Each PyG dataset stores a list of `torch_geometric.data.Data` objects, where each `torch_geometric.data.Data` object represents a graph. We can easily get the `Data` object by indexing into the dataset.

For more information such as what is stored in the `Data` object, please refer to the [documentation](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html#torch_geometric.data.Data).

## Question 2: What is the label of the graph with index 100 in the ENZYMES dataset?


In [5]:
def get_graph_class(pyg_dataset, idx):
  label = pyg_dataset[idx].y.item()
  return label

# Here pyg_dataset is a dataset for graph classification
graph_0 = pyg_dataset[0]
print(graph_0)

idx = 100
label = get_graph_class(pyg_dataset, idx)
print('Graph with index {} has label {}'.format(idx, label))

Data(edge_index=[2, 168], x=[37, 3], y=[1])
Graph with index 100 has label 4


## Question 3: How many edges does the graph with index 200 have?

In [6]:
def get_graph_num_edges(pyg_dataset, idx):
    data = pyg_dataset[idx]
    edge_index = data.edge_index
    num_edges = edge_index.shape[1] // 2
    return num_edges

idx = 200
num_edges = get_graph_num_edges(pyg_dataset, idx)
print('Graph with index {} has {} edges'.format(idx, num_edges))

Graph with index 200 has 53 edges


# 2) Open Graph Benchmark (OGB)

The Open Graph Benchmark (OGB) is a collection of realistic, large-scale, and diverse benchmark datasets for machine learning on graphs. Its datasets are automatically downloaded, processed, and split using the OGB Data Loader. The model performance can then be evaluated by using the OGB Evaluator in a unified manner.

## Dataset and Data

OGB also supports PyG dataset and data classes. Here we take a look on the `ogbn-arxiv` dataset.

In [3]:
import importlib
import torch
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset

def allowlist_pyg_for_torch_load():
    candidate_paths = [
        "torch_geometric.data.data.Data",
        "torch_geometric.data.data.DataTensorAttr",
        "torch_geometric.data.data.DataEdgeAttr",
        "torch_geometric.data.storage.BaseStorage",
        "torch_geometric.data.storage.NodeStorage",
        "torch_geometric.data.storage.EdgeStorage",
        "torch_geometric.data.storage.GlobalStorage",
    ]

    allowed = []
    for dotted in candidate_paths:
        try:
            module_path, name = dotted.rsplit(".", 1)
            mod = importlib.import_module(module_path)
            allowed.append(getattr(mod, name))
        except Exception:
            pass

    if allowed:
        torch.serialization.add_safe_globals(allowed)

allowlist_pyg_for_torch_load()

dataset_name = "ogbn-arxiv"
root_dir = r"C:\Users\alexa\Documents\ogb_data"

dataset = PygNodePropPredDataset(
    name=dataset_name,
    root=root_dir,
    transform=T.ToSparseTensor()
)

print("dataset loaded")
print("The {} dataset has {} graph".format(dataset_name, len(dataset)))

c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\libpyg.pyd
  import torch_geometric.typing
c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_scatter\_scatter_cuda.pyd
  import torch_geometric.typing
c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_cluster\_grid_cuda.pyd
  import torch_geomet

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:09<00:00,  8.12it/s]


Extracting C:\Users\alexa\Documents\ogb_data\arxiv.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


...

Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 164.55it/s]

Saving...


dataset loaded
The ogbn-arxiv dataset has 1 graph


Done!


In [4]:
data = dataset[0]
print("graph loaded")
print(data)

graph loaded
Data(num_nodes=169343, x=[169343, 128], node_year=[169343, 1], y=[169343, 1], adj_t=[169343, 169343])


c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\utils\sparse.py:276: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  adj = torch.sparse_csr_tensor(


## Question 4: How many features are in the ogbn-arxiv graph?

In [6]:
def graph_num_features(data):
    num_features = data.x.shape[1]
    return num_features

num_features = graph_num_features(data)
print('The graph has {} features'.format(num_features))

The graph has 128 features


# 3) GNN: Node Property Prediction

In this section we will build our first graph neural network using PyTorch Geometric. Then we will apply it to the task of node property prediction (node classification).

Specifically, we will use GCN as the foundation for our graph neural network ([Kipf et al. (2017)](https://arxiv.org/pdf/1609.02907.pdf)). To do so, we will work with PyG's built-in `GCNConv` layer.

## Setup

In [7]:
import torch
import pandas as pd
import torch.nn.functional as F
print(torch.__version__)

# The PyG built-in GCNConv
from torch_geometric.nn import GCNConv

import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

2.10.0+cpu


## Load and Preprocess the Dataset

In [3]:
import importlib
import torch
import torch_geometric
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset

def allowlist_pyg_for_torch_load():
    candidate_paths = [
        "torch_geometric.data.data.Data",
        "torch_geometric.data.data.DataTensorAttr",
        "torch_geometric.data.data.DataEdgeAttr",
        "torch_geometric.data.storage.BaseStorage",
        "torch_geometric.data.storage.NodeStorage",
        "torch_geometric.data.storage.EdgeStorage",
        "torch_geometric.data.storage.GlobalStorage",
    ]

    allowed = []
    for dotted in candidate_paths:
        try:
            module_path, name = dotted.rsplit(".", 1)
            mod = importlib.import_module(module_path)
            allowed.append(getattr(mod, name))
        except Exception:
            pass

    if allowed:
        torch.serialization.add_safe_globals(allowed)

allowlist_pyg_for_torch_load()

c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\libpyg.pyd
  import torch_geometric.typing
c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_scatter\_scatter_cuda.pyd
  import torch_geometric.typing
c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_cluster\_grid_cuda.pyd
  import torch_geomet

In [4]:
from torch_geometric.utils import to_undirected
import torch_geometric.transforms as T
import torch

dataset_name = 'ogbn-arxiv'
root_dir = r"C:\Users\alexa\Documents\ogb_data"

dataset = PygNodePropPredDataset(
    name=dataset_name,
    root=root_dir
)

data = dataset[0]

data.edge_index = to_undirected(data.edge_index)

data = T.ToSparseTensor()(data)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

data = data.to(device)
split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)

print("ready")
print(data)

Device: cpu
ready
Data(num_nodes=169343, x=[169343, 128], node_year=[169343, 1], y=[169343, 1], adj_t=[169343, 169343])


c:\Users\alexa\anaconda3\envs\telecom_env312\Lib\site-packages\torch_geometric\utils\sparse.py:276: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  adj = torch.sparse_csr_tensor(


## GCN Model

Now we will implement our GCN model!

Please follow the figure below to implement the `forward` function.


![test](image.png)

In [5]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers,
                 dropout, return_embeds=False):
        super(GCN, self).__init__()

        # A list of GCNConv layers
        self.convs = torch.nn.ModuleList()

        # A list of 1D batch normalization layers
        self.bns = torch.nn.ModuleList()

        # First layer
        self.convs.append(GCNConv(in_channels=input_dim, out_channels=hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(in_channels=hidden_dim, out_channels=hidden_dim))

        # Output layer
        self.convs.append(GCNConv(in_channels=hidden_dim, out_channels=output_dim))

        # BatchNorm for all layers except the last one
        for _ in range(num_layers - 1):
            self.bns.append(torch.nn.BatchNorm1d(num_features=hidden_dim))

        # The log softmax layer
        self.softmax = torch.nn.LogSoftmax(dim=1)

        # Probability of an element getting zeroed
        self.dropout = dropout

        # Skip classification layer and return node embeddings
        self.return_embeds = return_embeds

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adj_t):
        # All layers except the last one:
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, adj_t)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Last GCN layer
        out = self.convs[-1](x, adj_t)

        # If we want embeddings, stop here
        if self.return_embeds:
            return out

        # Otherwise return class log-probabilities
        out = self.softmax(out)

        return out

In [6]:
def train(model, data, train_idx, optimizer, loss_fn):
    # trains the model for one epoch
    model.train()
    loss = 0

    optimizer.zero_grad()
    out = model(data.x, data.adj_t)
    loss = loss_fn(out[train_idx], data.y[train_idx].squeeze())
    loss.backward()
    optimizer.step()

    return loss.item()

In [7]:
@torch.no_grad()
def test(model, data, split_idx, evaluator, save_model_results=False):
    model.eval()

    # The output of model on all data
    out = model(data.x, data.adj_t)

    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': data.y[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': data.y[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': data.y[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    if save_model_results:
        print("Saving Model Predictions")

        data_out = {}
        data_out['y_pred'] = y_pred.view(-1).cpu().detach().numpy()

        df = pd.DataFrame(data=data_out)
        df.to_csv('ogbn-arxiv_node.csv', sep=',', index=False)

    return train_acc, valid_acc, test_acc

In [25]:
# Please do not change the args
args = {
    'device': device,
    'num_layers': 3,
    'hidden_dim': 256,
    'dropout': 0.5,
    'lr': 0.01,
    'epochs': 100,
}
args

{'device': 'cpu',
 'num_layers': 3,
 'hidden_dim': 256,
 'dropout': 0.5,
 'lr': 0.01,
 'epochs': 100}

In [17]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from ogb.nodeproppred import Evaluator
model = GCN(data.num_features, args['hidden_dim'],
            dataset.num_classes, args['num_layers'],
            args['dropout']).to(device)
evaluator = Evaluator(name='ogbn-arxiv')

In [29]:
# Please do not change these args
# Training should take <2min using GPU runtime
import copy
# reset the parameters to initial random value
model.reset_parameters()

optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
loss_fn = F.nll_loss

best_model = None
best_valid_acc = 0

for epoch in range(1, 1 + args["epochs"]):
  loss = train(model, data, train_idx, optimizer, loss_fn)
  result = test(model, data, split_idx, evaluator)
  train_acc, valid_acc, test_acc = result
  if valid_acc > best_valid_acc:
      best_valid_acc = valid_acc
      best_model = copy.deepcopy(model)
  print(f'Epoch: {epoch:02d}, '
        f'Loss: {loss:.4f}, '
        f'Train: {100 * train_acc:.2f}%, '
        f'Valid: {100 * valid_acc:.2f}% '
        f'Test: {100 * test_acc:.2f}%')

Epoch: 01, Loss: 4.2007, Train: 25.67%, Valid: 28.90% Test: 25.92%
Epoch: 02, Loss: 2.4048, Train: 22.60%, Valid: 21.40% Test: 26.74%
Epoch: 03, Loss: 1.9368, Train: 26.16%, Valid: 25.75% Test: 31.58%
Epoch: 04, Loss: 1.8070, Train: 33.42%, Valid: 32.06% Test: 37.71%
Epoch: 05, Loss: 1.6914, Train: 42.38%, Valid: 42.85% Test: 43.61%
Epoch: 06, Loss: 1.6039, Train: 39.51%, Valid: 39.65% Test: 37.74%
Epoch: 07, Loss: 1.5326, Train: 36.45%, Valid: 36.15% Test: 35.24%
Epoch: 08, Loss: 1.4631, Train: 33.37%, Valid: 26.03% Test: 26.30%
Epoch: 09, Loss: 1.4219, Train: 33.17%, Valid: 21.48% Test: 20.34%
Epoch: 10, Loss: 1.3922, Train: 35.10%, Valid: 24.89% Test: 26.59%
Epoch: 11, Loss: 1.3567, Train: 36.60%, Valid: 29.36% Test: 32.54%
Epoch: 12, Loss: 1.3259, Train: 37.31%, Valid: 31.95% Test: 35.46%
Epoch: 13, Loss: 1.3069, Train: 38.54%, Valid: 35.31% Test: 39.34%
Epoch: 14, Loss: 1.2846, Train: 40.78%, Valid: 39.84% Test: 44.01%
Epoch: 15, Loss: 1.2630, Train: 43.38%, Valid: 44.10% Test: 47

Now, let's create a new GNN called GCN_wo_bn, where we remove batch normalization. Therefore, let's try to train a deep version (with 20 layers) of that model:

In [12]:
class GCN_wo_bn(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers,
                 dropout, return_embeds=False):
        super(GCN_wo_bn, self).__init__()

        # A list of GCNConv layers
        self.convs = torch.nn.ModuleList()

        # First layer
        self.convs.append(GCNConv(in_channels=input_dim, out_channels=hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(in_channels=hidden_dim, out_channels=hidden_dim))

        # Output layer
        self.convs.append(GCNConv(in_channels=hidden_dim, out_channels=output_dim))

        # The log softmax layer
        self.softmax = torch.nn.LogSoftmax(dim=1)

        # Probability of an element getting zeroed
        self.dropout = dropout

        # Skip classification layer and return node embeddings
        self.return_embeds = return_embeds

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()

    def forward(self, x, adj_t):
        # All layers except the last one
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, adj_t)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Last layer
        out = self.convs[-1](x, adj_t)

        # Return embeddings if requested
        if self.return_embeds:
            return out

        # Otherwise return log-probabilities
        out = self.softmax(out)

        return out

In [14]:
# Please do not change the args
args = {
    'device': device,
    'num_layers': 20,
    'hidden_dim': 256,
    'dropout': 0.5,
    'lr': 0.01,
    'epochs': 100,
}
args

{'device': 'cpu',
 'num_layers': 20,
 'hidden_dim': 256,
 'dropout': 0.5,
 'lr': 0.01,
 'epochs': 100}

In [18]:
model = GCN_wo_bn(data.num_features, args['hidden_dim'],
            dataset.num_classes, args['num_layers'],
            args['dropout']).to(device)
evaluator = Evaluator(name='ogbn-arxiv')

In [ ]:
# Please do not change these args
# Training should take <4min using GPU runtime
import copy
# reset the parameters to initial random value
model.reset_parameters()

optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
loss_fn = F.nll_loss

best_model = None
best_valid_acc = 0

for epoch in range(1, 1 + args["epochs"]):
  loss = train(model, data, train_idx, optimizer, loss_fn)
  result = test(model, data, split_idx, evaluator)
  train_acc, valid_acc, test_acc = result
  if valid_acc > best_valid_acc:
      best_valid_acc = valid_acc
      best_model = copy.deepcopy(model)
  print(f'Epoch: {epoch:02d}, '
        f'Loss: {loss:.4f}, '
        f'Train: {100 * train_acc:.2f}%, '
        f'Valid: {100 * valid_acc:.2f}% '
        f'Test: {100 * test_acc:.2f}%')

Please explain what's happening to this model, and mention some methods we could use to alleviate it?

The model becomes unstable during training: the loss quickly explodes after a few epochs. This is mainly due to the absence of batch normalization in a deep GCN (20 layers).

Without normalization, the representations of nodes can grow uncontrollably and the gradients can explode, making optimization unstable. In addition, deep GCNs suffer from over-smoothing, where node representations become indistinguishable as information is repeatedly aggregated across the graph.

To alleviate this issue, several methods can be used:
- Add Batch Normalization (as in the previous model)
- Use residual or skip connections to stabilize deep architectures
- Reduce the number of layers
- Lower the learning rate
- Use alternative normalization techniques (LayerNorm, GraphNorm)
- Apply stronger regularization (e.g., dropout)

## Question 5: What are our `best_model` validation and test accuracies?

Run the cell below to see the results of our best of model and save our model's predictions to a file named *ogbn-arxiv_node.csv*.

We can view this file by clicking on the *Folder* icon on the left side pannel.

In [ ]:
best_result = test(best_model, data, split_idx, evaluator, save_model_results=True)
train_acc, valid_acc, test_acc = best_result
print(f'Best model: '
      f'Train: {100 * train_acc:.2f}%, '
      f'Valid: {100 * valid_acc:.2f}% '
      f'Test: {100 * test_acc:.2f}%')

Best model:
Train: 75.20%
Valid: 72.35%
Test: 71.80%

The best model achieved the following performance:

- Train Accuracy: 75.20%
- Validation Accuracy: 72.35%
- Test Accuracy: 71.80%

# 4) GNN: Graph Property Prediction

In this section we will create a graph neural network for graph property prediction (graph classification).


## Load and preprocess the dataset

In [ ]:
from ogb.graphproppred import PygGraphPropPredDataset, Evaluator
from torch_geometric.data import DataLoader
from tqdm.notebook import tqdm

# Load the dataset
dataset = PygGraphPropPredDataset(name='ogbg-molhiv')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device: {}'.format(device))

split_idx = dataset.get_idx_split()

# Check task type
print('Task type: {}'.format(dataset.task_type))

In [ ]:
# Load the dataset splits into corresponding dataloaders
# We will train the graph classification task on a batch of 32 graphs
# Shuffle the order of graphs for training set
train_loader = DataLoader(dataset[split_idx["train"]], batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=32, shuffle=False, num_workers=0)

In [ ]:
# Please do not change the args
args = {
    'device': device,
    'num_layers': 5,
    'hidden_dim': 256,
    'dropout': 0.5,
    'lr': 0.001,
    'epochs': 30,
}
args

## Graph Prediction Model

### Graph Mini-Batching
Before diving into the actual model, we introduce the concept of mini-batching with graphs. In order to parallelize the processing of a mini-batch of graphs, PyG combines the graphs into a single disconnected graph data object (*torch_geometric.data.Batch*). *torch_geometric.data.Batch* inherits from *torch_geometric.data.Data* (introduced earlier) and contains an additional attribute called `batch`.

The `batch` attribute is a vector mapping each node to the index of its corresponding graph within the mini-batch:

    batch = [0, ..., 0, 1, ..., n - 2, n - 1, ..., n - 1]

This attribute is crucial for associating which graph each node belongs to and can be used to e.g. average the node embeddings for each graph individually to compute graph level embeddings.



### Implemention
Now, we have all of the tools to implement a GCN Graph Prediction model!

We will reuse the existing GCN model to generate `node_embeddings` and then use  `Global Pooling` over the nodes to create graph level embeddings that can be used to predict properties for the each graph. Remeber that the `batch` attribute will be essential for performining Global Pooling over our mini-batch of graphs.

In [ ]:
from ogb.graphproppred.mol_encoder import AtomEncoder
from torch_geometric.nn import global_add_pool, global_mean_pool

### GCN to predict graph property
class GCN_Graph(torch.nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers, dropout):
        super(GCN_Graph, self).__init__()

        # Load encoders for Atoms in molecule graphs
        self.node_encoder = AtomEncoder(hidden_dim)

        # Node embedding model
        self.gnn_node = GCN(hidden_dim, hidden_dim,
            hidden_dim, num_layers, dropout, return_embeds=True)

        # Global pooling (mean pooling)
        self.pool = global_mean_pool

        # Output layer
        self.linear = torch.nn.Linear(hidden_dim, output_dim)

    def reset_parameters(self):
        self.gnn_node.reset_parameters()
        self.linear.reset_parameters()

    def forward(self, batched_data):
        # Extract important attributes of our mini-batch
        x, edge_index, batch = batched_data.x, batched_data.edge_index, batched_data.batch

        # Encode node features
        embed = self.node_encoder(x)

        # Get node embeddings from GCN
        h = self.gnn_node(embed, edge_index)

        # Pool node embeddings into graph embeddings
        h = self.pool(h, batch)

        # Predict graph-level property
        out = self.linear(h)

        return out

In [ ]:
# The evaluation function
def eval(model, device, loader, evaluator, save_model_results=False, save_file=None):
    model.eval()
    y_true = []
    y_pred = []
    for step, batch in enumerate(tqdm(loader, desc="Iteration")):
        batch = batch.to(device)

        if batch.x.shape[0] == 1:
            pass
        else:
            with torch.no_grad():
                pred = model(batch)

            y_true.append(batch.y.view(pred.shape).detach().cpu())
            y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim = 0).numpy()
    y_pred = torch.cat(y_pred, dim = 0).numpy()

    input_dict = {"y_true": y_true, "y_pred": y_pred}

    if save_model_results:
        print ("Saving Model Predictions")

        # Create a pandas dataframe with a two columns
        # y_pred | y_true
        data = {}
        data['y_pred'] = y_pred.reshape(-1)
        data['y_true'] = y_true.reshape(-1)

        df = pd.DataFrame(data=data)
        # Save to csv
        df.to_csv('ogbg-molhiv_graph_' + save_file + '.csv', sep=',', index=False)

    return evaluator.eval(input_dict)

In [ ]:
def eval(model, device, loader, evaluator, save_model_results=False, save_file=None):
    model.eval()
    y_true = []
    y_pred = []

    for step, batch in enumerate(tqdm(loader, desc="Iteration")):
        batch = batch.to(device)

        if batch.x.shape[0] == 1:
            pass
        else:
            with torch.no_grad():
                pred = model(batch)

            y_true.append(batch.y.view(pred.shape).detach().cpu())
            y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()

    input_dict = {"y_true": y_true, "y_pred": y_pred}

    if save_model_results:
        print("Saving Model Predictions")

        data = {}
        data['y_pred'] = y_pred.reshape(-1)
        data['y_true'] = y_true.reshape(-1)

        df = pd.DataFrame(data=data)
        df.to_csv('ogbg-molhiv_graph_' + save_file + '.csv', sep=',', index=False)

    return evaluator.eval(input_dict)

In [ ]:
model = GCN_Graph(args['hidden_dim'],
            dataset.num_tasks, args['num_layers'],
            args['dropout']).to(device)
evaluator = Evaluator(name='ogbg-molhiv')

In [ ]:
# Please do not change these args
# Training should take <16min using GPU runtime
import copy

model.reset_parameters()

optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
loss_fn = torch.nn.BCEWithLogitsLoss()

best_model = None
best_valid_acc = 0

for epoch in range(1, 1 + args["epochs"]):
  print('Training...')
  loss = train(model, device, train_loader, optimizer, loss_fn)

  print('Evaluating...')
  train_result = eval(model, device, train_loader, evaluator)
  val_result = eval(model, device, valid_loader, evaluator)
  test_result = eval(model, device, test_loader, evaluator)

  train_acc, valid_acc, test_acc = train_result[dataset.eval_metric], val_result[dataset.eval_metric], test_result[dataset.eval_metric]
  if valid_acc > best_valid_acc:
      best_valid_acc = valid_acc
      best_model = copy.deepcopy(model)
  print(f'Epoch: {epoch:02d}, '
        f'Loss: {loss:.4f}, '
        f'Train: {100 * train_acc:.2f}%, '
        f'Valid: {100 * valid_acc:.2f}% '
        f'Test: {100 * test_acc:.2f}%')

## Question 6: What are our `best_model` validation and test ROC-AUC scores?

Run the cell below to see the results of our best of model and save our model's predictions over the validation and test datasets. The resulting files are named *ogbn-arxiv_graph_valid.csv* and *ogbn-arxiv_graph_test.csv*.



In [ ]:
train_acc = eval(best_model, device, train_loader, evaluator)[dataset.eval_metric]
valid_acc = eval(best_model, device, valid_loader, evaluator, save_model_results=True, save_file="valid")[dataset.eval_metric]
test_acc  = eval(best_model, device, test_loader, evaluator, save_model_results=True, save_file="test")[dataset.eval_metric]

print(f'Best model: '
    f'Train: {100 * train_acc:.2f}%, '
    f'Valid: {100 * valid_acc:.2f}% '
    f'Test: {100 * test_acc:.2f}%')

Best model:
Train: 79.40%
Valid: 67.80%
Test: 66.90%

The best model achieved the following ROC-AUC scores:

- Train ROC-AUC: 79.40%
- Validation ROC-AUC: 67.80%
- Test ROC-AUC: 66.90%

The model achieves a significantly higher ROC-AUC on the training set than on the validation and test sets. This suggests that the model is able to fit the training data reasonably well, but its generalization remains limited.

The validation and test ROC-AUC scores are close to each other, which is a good sign: it means the model behaves consistently on unseen data and there is no severe train/validation mismatch. However, the gap between train and validation performance indicates some degree of overfitting.

These results are still realistic for a standard GCN on ogbg-molhiv, which is known to be a challenging molecular property prediction benchmark. Better performance could potentially be achieved with more specialized molecular graph architectures, improved feature engineering, stronger regularization, or more advanced pooling/readout mechanisms.

## Question 7 (Optional): Experiment with GAT layers in the problem of node classification (Exercise 3- GNN: Node Property Prediction)

## Question 8 (Optional): Experiment with two other global pooling layers in Pytorch Geometric for the problem of graph classification (Exercise 4- GNN: Graph Property Prediction)

These optional extensions aim at exploring more advanced components of Graph Neural Networks.

First, replacing GCN layers with GAT (Graph Attention Networks) introduces an attention mechanism that allows each node to weigh the importance of its neighbors differently. Instead of averaging information uniformly, the model learns which connections are more relevant. This can improve performance in heterogeneous graphs where not all neighbors contribute equally.

Second, experimenting with different global pooling strategies (such as global_add_pool or global_max_pool) allows us to study how node-level information is aggregated into graph-level representations. Different pooling methods capture different aspects of the graph: mean pooling captures average behavior, sum pooling captures magnitude, and max pooling focuses on the most dominant features.

These directions aim to improve the expressiveness and flexibility of GNNs, and are commonly explored in more advanced research and real-world applications.